# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** 用户数据_清洗后.csv（使用之前生成的清洗数据）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [ ]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
    Path("用户数据_清洗后.csv"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到数据文件，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent, Path("/Users/yq/Desktop/muc")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if DATA_PATH.suffix == ".xlsx":
    raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
else:
    raw_df = pd.read_csv(DATA_PATH)

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [ ]:
# 在此写下你的答案：
# 1. 每条记录代表一个电商平台用户的详细信息和行为数据
# 2. 项目的目标变量是 Churn 列，表示用户是否流失（Yes/No）
# 3. CustomerID 是用户唯一标识，仅用于区分不同用户，不包含业务含义，
#    作为连续数值参与分析会引入无意义的数值关系（如 ID 大小比较），
#    可能导致模型学习到虚假的模式

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [ ]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    # TODO：返回一个 DataFrame，至少包含：
    # 数据类型、缺失数量、缺失比例(%)、唯一值数量
    report = pd.DataFrame({
        "数据类型": data.dtypes,
        "缺失数量": data.isna().sum(),
        "缺失比例(%)": (data.isna().mean() * 100).round(2),
        "唯一值数量": data.nunique()
    })
    return report

# TODO：生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [ ]:
# TODO：完成项目初始审计
print("完全重复行数：", raw_df.duplicated().sum())
print("CustomerID 重复数量：", raw_df["CustomerID"].duplicated().sum())

churn_counts = raw_df["Churn"].value_counts()
print("\nChurn 频数：")
print(churn_counts)
churn_rate = churn_counts.get("Yes", 0) / churn_counts.sum() * 100
print(f"流失率：{churn_rate:.2f}%")

for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())

---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [ ]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone",
        "Mobile": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [ ]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # TODO：复制数据，避免覆盖原始数据
    cleaned_df = data.copy()
    logs = []
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # TODO：删除完全重复行，并记录日志
    before_rows = len(cleaned_df)
    cleaned_df = cleaned_df.drop_duplicates()
    after_rows = len(cleaned_df)
    affected = before_rows - after_rows
    logs.append({
        "处理时间": timestamp,
        "处理步骤": "删除完全重复行",
        "处理规则": "删除所有完全相同的重复记录",
        "处理前记录数": before_rows,
        "处理后记录数": after_rows,
        "影响记录数": affected,
        "备注": ""
    })

    # TODO：对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    for col in NUMERIC_MISSING_COLS:
        before_missing = cleaned_df[col].isna().sum()
        if before_missing > 0:
            median_val = cleaned_df[col].median()
            cleaned_df[col] = cleaned_df[col].fillna(median_val)
            after_missing = cleaned_df[col].isna().sum()
            logs.append({
                "处理时间": timestamp,
                "处理步骤": f"中位数填补缺失值",
                "处理规则": f"{col} 使用总体中位数 {median_val:.2f} 填补",
                "处理前记录数": before_missing,
                "处理后记录数": after_missing,
                "影响记录数": before_missing - after_missing,
                "备注": "不按 Churn 分组填补"
            })

    # TODO：对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    for col, mappings in CATEGORY_MAPPINGS.items():
        for old_val, new_val in mappings.items():
            before_count = (cleaned_df[col] == old_val).sum()
            if before_count > 0:
                cleaned_df[col] = cleaned_df[col].replace(old_val, new_val)
                after_count = (cleaned_df[col] == old_val).sum()
                logs.append({
                    "处理时间": timestamp,
                    "处理步骤": "类别标准化",
                    "处理规则": f"{col} 中 {old_val} → {new_val}",
                    "处理前记录数": before_count,
                    "处理后记录数": after_count,
                    "影响记录数": before_count - after_count,
                    "备注": "统一同义类别"
                })

    # TODO：将 Churn 和 Complain 转为整数类型
    for col in ["Churn", "Complain"]:
        cleaned_df[col] = cleaned_df[col].map({"Yes": 1, "No": 0})
        logs.append({
            "处理时间": timestamp,
            "处理步骤": "类型转换",
            "处理规则": f"{col} Yes→1, No→0",
            "处理前记录数": len(cleaned_df),
            "处理后记录数": len(cleaned_df),
            "影响记录数": len(cleaned_df),
            "备注": "转为整数类型便于后续分析"
        })

    # TODO：返回 cleaned_df 与 cleaning_log
    cleaning_log = pd.DataFrame(logs)
    return cleaned_df, cleaning_log

### 任务 3：运行清洗函数并查看日志

In [ ]:
# TODO：执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
cleaned_df.head()

---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [ ]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# TODO：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
tenure_bins = [-1, 0, 12, 24, 36, 48, 60, 72, float("inf")]
tenure_labels = ["0个月", "1-12个月", "13-24个月", "25-36个月", "37-48个月", "49-60个月", "61-72个月", "72个月以上"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels)

# TODO：新建 IsMobileLogin，移动端为 1，其他设备为 0
cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)

# TODO：生成 outlier_report（每行对应一个待检查字段）
outlier_report = pd.DataFrame([
    iqr_outlier_summary(cleaned_df["WarehouseToHome"]),
    iqr_outlier_summary(cleaned_df["OrderCount"]),
    iqr_outlier_summary(cleaned_df["CashbackAmount"])
], index=["WarehouseToHome", "OrderCount", "CashbackAmount"])

print("TenureGroup 分布：")
print(cleaned_df["TenureGroup"].value_counts())

print("\nIsMobileLogin 分布：")
print(cleaned_df["IsMobileLogin"].value_counts())

print("\n候选异常值报告：")
display(outlier_report)

### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [ ]:
# TODO：完成业务规则检查
business_rule_report = pd.DataFrame({
    "规则": [
        "使用时长小于 0",
        "仓库距离小于 0",
        "订单数小于或等于 0",
        "返现金额小于 0"
    ],
    "不合规记录数": [
        (cleaned_df["HourSpendOnApp"] < 0).sum(),
        (cleaned_df["WarehouseToHome"] < 0).sum(),
        (cleaned_df["OrderCount"] <= 0).sum(),
        (cleaned_df["CashbackAmount"] < 0).sum()
    ]
})
display(business_rule_report)

# 处理结论：
# 本次检查中所有业务规则不合规记录数均为 0，说明数据在业务规则层面质量良好。
# 无需针对业务规则违规进行额外处理。

---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [ ]:
# TODO：完成最终验收
quality_after = build_quality_report(cleaned_df)

assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值字段仍有缺失值"
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "登录设备尚未统一"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式尚未统一"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式尚未统一"
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "新增字段缺失"

print("数据清洗验收通过！")

# TODO：导出下列文件，使用 utf-8-sig 编码：
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

# TODO：输出 outlier_report 和 business_rule_report
print("\n候选异常值报告：")
display(outlier_report)

print("业务规则检查报告：")
display(business_rule_report)

# TODO：输出交付文件的路径
print("\n交付文件路径：")
print(f"  - 清洗后数据：{OUTPUT_DIR / 'ecommerce_customer_cleaned.csv'}")
print(f"  - 清洗前质量报告：{OUTPUT_DIR / 'data_quality_before.csv'}")
print(f"  - 清洗后质量报告：{OUTPUT_DIR / 'data_quality_after.csv'}")
print(f"  - 清洗日志：{OUTPUT_DIR / 'cleaning_log.csv'}")

## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

In [ ]:
# 1. 本项目发现的数据质量问题：
#    - 多个数值字段存在约10%的缺失值（Tenure、WarehouseToHome等）
#    - 类别字段存在同义不同名问题（Phone/Mobile Phone、COD/Cash on Delivery等）
#    - 无完全重复行和业务规则违规记录

# 2. 处理策略：
#    - 缺失值：使用总体中位数填充，避免均值受极端值影响
#    - 类别不一致：统一同义类别，确保数据一致性
#    - 候选异常值：仅记录不删除，需结合业务判断

# 3. 清洗后数据可作为第五天分析输入的原因：
#    - 数值字段无缺失，类别字段已标准化，新增了TenureGroup和IsMobileLogin特征
#    - 目标变量Churn和Complain已转为整数类型，便于建模
#    - 清洗过程可追溯，日志完整

# 4. 需要业务人员确认的处理规则：
#    - 中位数填充是否适用于所有数值缺失场景
#    - IQR识别的候选异常值是否为真实异常
#    - 类别统一规则是否符合业务实际含义